In [ ]:
import pandas as pd
from google.colab import drive



# 1. Load the file directly from your Google Drive
# (Change 'My Drive/Folder_Name/' if you put it inside a specific folder)
file_path = '/content/drive/MyDrive/csc490/train-00000-of-00001.parquet'
df = pd.read_parquet(file_path)

# 2. Verify it loaded perfectly by checking the column names
print("Successfully loaded columns:", df.columns.tolist())
print(f"Total rows found: {len(df)}")


Successfully loaded columns: ['image', 'text']
Total rows found: 484


In [13]:
import os
import json
import random
import shutil
import asyncio
from bs4 import BeautifulSoup

# Ensure Playwright dependencies are set up
print("Verifying Playwright system binaries...")
os.system("pip install -q playwright beautifulsoup4")
os.system("playwright install --with-deps chromium")

from playwright.async_api import async_playwright

OUTPUT_DIR = "/content/design2code_mutated"
IMAGES_DIR = os.path.join(OUTPUT_DIR, "screenshots")
METADATA_DIR = os.path.join(OUTPUT_DIR, "metadata")
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(METADATA_DIR, exist_ok=True)

# ==========================================
# DEFINE THE MUTATION ENGINE
# ==========================================
def mutate_typography(soup):
    is_regression = random.choice([True, False])
    body = soup.find('body') or soup
    style_attr = body.get('style', '')
    if is_regression:
        body['style'] = style_attr + "; font-size: 2px !important; font-family: 'Wingdings' !important;"
        label = "regression"
    else:
        body['style'] = style_attr + "; font-family: 'Courier New', monospace !important; line-height: 1.6 !important;"
        label = "benign"
    return str(soup), label

def mutate_color(soup):
    is_regression = random.choice([True, False])
    body = soup.find('body') or soup
    style_attr = body.get('style', '')
    if is_regression:
        body['style'] = style_attr + "; background-color: #000000 !important; color: #111111 !important;"
        label = "regression"
    else:
        body['style'] = style_attr + "; background-color: #f4f4f9 !important;"
        label = "benign"
    return str(soup), label

def mutate_layout(soup):
    is_regression = random.choice([True, False])
    containers = soup.find_all(['div', 'section', 'header', 'footer']) or [soup]
    label = "benign"
    if is_regression:
        label = "regression"
        for idx, el in enumerate(containers):
            if idx % 3 == 0:
                el['style'] = el.get('style', '') + "; display: block !important; position: absolute !important; top: 0 !important;"
    else:
        for el in containers:
            style = el.get('style', '')
            if 'display: flex' in style or 'display:flex' in style:
                el['style'] = style + "; flex-direction: column-reverse !important;"
    return str(soup), label

def mutate_spacing(soup):
    is_regression = random.choice([True, False])
    all_elements = soup.find_all(['div', 'p', 'h1', 'section']) or [soup]
    if is_regression:
        label = "regression"
        for el in random.sample(all_elements, min(len(all_elements), 5)):
            el['style'] = el.get('style', '') + "; margin: 150px !important; padding: 100px !important;"
    else:
        label = "benign"
        for el in random.sample(all_elements, min(len(all_elements), 5)):
            el['style'] = el.get('style', '') + "; padding: 20px !important;"
    return str(soup), label

def mutate_deletion(soup):
    elements = soup.find_all(['header', 'footer', 'nav', 'h1', 'button', 'img'])
    if elements:
        for target in random.sample(elements, min(len(elements), 2)):
            target.decompose()
    return str(soup), "regression"

def mutate_sizing(soup):
    is_regression = random.choice([True, False])
    elements = soup.find_all(['div', 'img', 'section']) or [soup]
    if is_regression:
        label = "regression"
        for el in random.sample(elements, min(len(elements), 5)):
            el['style'] = el.get('style', '') + "; width: 1px !important; overflow: hidden !important;"
    else:
        label = "benign"
        for el in random.sample(elements, min(len(elements), 3)):
            el['style'] = el.get('style', '') + "; max-width: 90% !important; margin: 0 auto !important;"
    return str(soup), label

def mutate_visibility(soup):
    is_regression = random.choice([True, False])
    elements = soup.find_all(['p', 'button', 'h1', 'div']) or [soup]
    if is_regression:
        label = "regression"
        for el in random.sample(elements, min(len(elements), 5)):
            el['style'] = el.get('style', '') + "; opacity: 0.01 !important; filter: blur(10px) !important;"
    else:
        label = "benign"
        for el in random.sample(elements, min(len(elements), 3)):
            el['style'] = el.get('style', '') + "; opacity: 0.90 !important;"
    return str(soup), label

MUTATION_MAP = {
    "typography": mutate_typography,
    "colour": mutate_color,
    "layout": mutate_layout,
    "spacing": mutate_spacing,
    "deletion": mutate_deletion,
    "sizing": mutate_sizing,
    "visibility": mutate_visibility
}

# ==========================================
# PIPELINE RUNNING DIRECTLY FROM PANDAS DF
# ==========================================
async def run_pipeline(dataframe):
    print(f"Starting processing directly from loaded dataframe with {len(dataframe)} rows...")
    metadata_records = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page(viewport={"width": 1280, "height": 800})

        for index, row in dataframe.iterrows():
            parent_id = f"d2c_{index:03d}"
            raw_html = row.get('text')

            if not raw_html:
                continue

            print(f"Mutating item [{index + 1}/{len(dataframe)}] (ID: {parent_id})...")

            for mutation_name, mutation_func in MUTATION_MAP.items():
                mutation_id = f"{parent_id}_{mutation_name}"
                soup_instance = BeautifulSoup(raw_html, "html.parser")
                mutated_html, label_classification = mutation_func(soup_instance)

                try:
                    await page.set_content(mutated_html, wait_until="load")
                    await page.wait_for_timeout(100)

                    screenshot_filename = f"{mutation_id}.png"
                    screenshot_path = os.path.join(IMAGES_DIR, screenshot_filename)
                    await page.screenshot(path=screenshot_path, full_page=True)

                    record = {
                        "mutation_id": mutation_id,
                        "parent_id": parent_id,
                        "mutation_type": mutation_name,
                        "label": label_classification,
                        "screenshot_path": screenshot_path,
                        "mutated_html": mutated_html
                    }
                    metadata_records.append(record)

                    with open(os.path.join(METADATA_DIR, f"{mutation_id}.json"), "w") as f:
                        json.dump(record, f, indent=4)

                except Exception as e:
                    print(f"Warning: Render bypass on {mutation_id}: {e}")
                    continue

        await browser.close()

    master_json_path = os.path.join(OUTPUT_DIR, "master_dataset.json")
    with open(master_json_path, "w") as f:
        json.dump(metadata_records, f, indent=4)

    print(f"\nFinished processing! Total items generated: {len(metadata_records)}")

    if len(metadata_records) > 0:
        print("Saving generated folder structure as a zip file...")
        shutil.make_archive(
            base_name="/content/design2code_mutated",
            format="zip",
            root_dir=OUTPUT_DIR
        )
        print("Success! The structural zip archive has been compiled.")

        from google.colab import files
        print("Downloading zip package to local machine...")
        files.download('/content/design2code_mutated.zip')
    else:
        print("Error: No mutations generated. Verify dataframe strings.")

# Pass your active dataframe directly into the pipeline function
await run_pipeline(df)


Verifying Playwright system binaries...
Starting processing directly from loaded dataframe with 484 rows...
Mutating item [1/484] (ID: d2c_000)...
Mutating item [2/484] (ID: d2c_001)...
Mutating item [3/484] (ID: d2c_002)...
Mutating item [4/484] (ID: d2c_003)...
Mutating item [5/484] (ID: d2c_004)...
Mutating item [6/484] (ID: d2c_005)...
Mutating item [7/484] (ID: d2c_006)...
Mutating item [8/484] (ID: d2c_007)...
Mutating item [9/484] (ID: d2c_008)...
Mutating item [10/484] (ID: d2c_009)...
Mutating item [11/484] (ID: d2c_010)...
Mutating item [12/484] (ID: d2c_011)...
Mutating item [13/484] (ID: d2c_012)...
Mutating item [14/484] (ID: d2c_013)...
Mutating item [15/484] (ID: d2c_014)...
Mutating item [16/484] (ID: d2c_015)...
Mutating item [17/484] (ID: d2c_016)...
Mutating item [18/484] (ID: d2c_017)...
Mutating item [19/484] (ID: d2c_018)...
Mutating item [20/484] (ID: d2c_019)...
Mutating item [21/484] (ID: d2c_020)...
Mutating item [22/484] (ID: d2c_021)...
Mutating item [23/484

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>